# Upper Confidence Bound (UCB) based agents

> Agents utelizing the UCB based approach for Dynamic pricing and learning problems from https://doi.org/10.48550/arXiv.1604.07463

In [ ]:
#| default_exp agents.dynamic_pricing.UCB

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction


In [ ]:
#| export
class UCBPolicy():
    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = None,
                 actionprocessors: Optional[List[object]] = None,
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        assert type(alpha) == type(beta), "alpha and beta must be of the same type"
        if alpha is None:
            alpha = np.zeros(environment_info.observation_space.shape[0])
            beta = np.zeros(environment_info.observation_space.shape[0])
        if isinstance(ex_prices, list):
            ex_prices = np.array(ex_prices)
        assert ex_prices.shape[0] >= 2

        self.environment_info = environment_info
        self.ex_prices = ex_prices
        self.alpha = alpha
        self.beta = beta
        self.actionprocessors = actionprocessors
        self.price_function = price_function # Needs to return an np array
        self.lam = lam
        self.reg = reg
        self.g = g
        self.t = 0
        self.X = np.empty((0, environment_info.observation_space.shape[0] * 2))
        self.Y = np.empty((0, 1))
        self.mode = "train"
        self.actionprocessors.append(ClipAction(environment_info.action_space.low, environment_info.action_space.high))

    def draw_action(self, observation: np.ndarray):
        if self.t in [0, 1]:
            price = self.ex_prices[self.t]
        else:
            M = self.compute_uncertainty_M(observation)
            samples = self.sample_from_confidence_region(np.concatenate([self.alpha, self.beta]), M)
            alpha, beta = self.max_rev(samples, observation)
            price = self.price_function(alpha, beta, observation)
            
        for processor in self.actionprocessors:
            price = processor(price)
        
        return price
    
    def sample_design_matrix(self):
        I = np.identity(2*self.environment_info.observation_space.shape[0])
        I_lamdba = self.lam * I
        if self.X.shape[0] == 0:
            return I_lamdba
        matrix = np.sum([np.outer(x, x.T) for x in self.X], axis=0)
        return I_lamdba + matrix
    
    def sample_from_confidence_region(self, theta_hat, M, N=50):
        L = np.linalg.cholesky(np.linalg.inv(M))
        u = np.random.randn(len(theta_hat), N)
        u /= np.linalg.norm(u, axis=0)
        samples = theta_hat[:, np.newaxis] + 1/self.environment_info.observation_space.shape[0] * L @ u
        return samples.T
    
    def compute_uncertainty_M(self, x_t):
        M = self.sample_design_matrix()
        block_matrix = np.block([
            [x_t, np.zeros_like(x_t)],
            [np.zeros_like(x_t), x_t]
        ])
        M_inverse = np.linalg.inv(M)

        projected_matrix = block_matrix @ M_inverse @ block_matrix.T
        projected_matrix_inverse = np.linalg.inv(projected_matrix)
        return projected_matrix_inverse
    
    def fit(self, X, Y, action):
        assert self.mode == "train"
        self.t += 1
        X = np.concatenate([X, X * action], axis=1)
        self.X = np.vstack([self.X, X])
        self.Y = np.vstack([self.Y, Y])
        self.parameter_update()
    
    def parameter_update(self):
        model = sm.GLM(self.Y, self.X, family=sm.families.Gamma())
        results = model.fit()
        self.alpha = results.params[self.environment_info.observation_space.shape[0]:]
        self.beta = results.params[:self.environment_info.observation_space.shape[0]]
        
    def reset(self):
        return

In [ ]:
#| export
class UCBCoreAgent(Agent):

    """
    Base class for UCB agents.
    """

    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = UCBPolicy(lam=lam, reg=reg, environment_info=environment_info, obsprocessors=obsprocessors, actionprocessors=actionprocessors, ex_prices=ex_prices, alpha=alpha, beta=beta, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0][0]
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)

In [ ]:
#| export
class UCBAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for UCBCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 lam: float,
                 reg: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = UCBCoreAgent(lam=lam, reg=reg, environment_info=environment_info,
                                  obsprocessors=obsprocessors, 
                                  actionprocessors=actionprocessors, 
                                  agent_name=agent_name, 
                                  ex_prices=ex_prices, 
                                  alpha=alpha, 
                                  beta=beta, 
                                  price_function=price_function, 
                                  g=g)
        super().__init__(environment_info=environment_info, obsprocessors=obsprocessors, agent_name=agent_name)